In [2]:
import pandas as pd

In [ ]:
import pandas as pd
import os
from tqdm import tqdm

# Path to root folder with patient folders like '94956/'
root_dir = "/Users/william/Desktop/ERP/mimic3-benchmarks/data/root"

# Path to ITEMID ↔ LABEL reference file
d_items_path = "/Users/william/Desktop/ERP/Data/D_ITEMS.csv"
d_items = pd.read_csv(d_items_path)[['ITEMID', 'LABEL']]
d_items['LABEL'] = d_items['LABEL'].str.replace(r'\s+', '_', regex=True)

# Output folder for per-patient files
output_dir = "/Users/william/Desktop/ERP/Data/Stage_2"
os.makedirs(output_dir, exist_ok=True)

# Iterate through each patient folder
for subject_id in tqdm(os.listdir(root_dir)):
    subject_path = os.path.join(root_dir, subject_id)
    if not os.path.isdir(subject_path):
        continue

    events_path = os.path.join(subject_path, "events.csv")
    stays_path = os.path.join(subject_path, "stays.csv")

    if not os.path.exists(events_path) or not os.path.exists(stays_path):
        continue

    try:
        # Load event and stay data
        events = pd.read_csv(events_path, low_memory=False)
        stays = pd.read_csv(stays_path, low_memory=False)

        # Parse datetime columns
        events['CHARTTIME'] = pd.to_datetime(events['CHARTTIME'], errors='coerce')
        stays['ADMITTIME'] = pd.to_datetime(stays['ADMITTIME'], errors='coerce')

        # Map ITEMID to LABEL
        events = events.merge(d_items, on='ITEMID', how='left')
        events.dropna(subset=["LABEL", "CHARTTIME"], inplace=True)
        events['VALUE'] = pd.to_numeric(events['VALUE'], errors='coerce')
        events.dropna(subset=["VALUE"], inplace=True)

        # Floor CHARTTIME to nearest hour
        events['CHARTTIME_HOUR'] = events['CHARTTIME'].dt.floor('h')

        # Group and compute mean VALUE per label per hour (include ICUSTAY_ID)
        grouped = (
            events.groupby(['SUBJECT_ID', 'HADM_ID', 'ICUSTAY_ID', 'CHARTTIME_HOUR', 'LABEL'])['VALUE']
            .mean()
            .reset_index()
        )

        # Pivot to wide format
        pivoted = grouped.pivot_table(
            index=['SUBJECT_ID', 'HADM_ID', 'ICUSTAY_ID', 'CHARTTIME_HOUR'],
            columns='LABEL',
            values='VALUE'
        ).reset_index()

        # Add LOS, GENDER, MORTALITY info from stays
        pivoted = pivoted.merge(
            stays[['SUBJECT_ID', 'HADM_ID', 'LOS', 'GENDER', 'MORTALITY_INUNIT', 'MORTALITY_INHOSPITAL',"AGE"]],
            on=['SUBJECT_ID', 'HADM_ID'],
            how='left'
        )

        # Rename CHARTTIME_HOUR to CHARTTIME
        pivoted.rename(columns={'CHARTTIME_HOUR': 'CHARTTIME'}, inplace=True)

        # Reorder for readability
        front_cols = ['SUBJECT_ID', 'HADM_ID', 'ICUSTAY_ID', 'CHARTTIME', 'LOS', "AGE", 'GENDER', 'MORTALITY_INUNIT', 'MORTALITY_INHOSPITAL']
        other_cols = [col for col in pivoted.columns if col not in front_cols]
        pivoted = pivoted[front_cols + other_cols]

        # Save per-patient file
        save_path = os.path.join(output_dir, f"{subject_id}.csv")
        pivoted.to_csv(save_path, index=False)

    except Exception as e:
        print(f"Error processing {subject_id}: {e}")


  0%|          | 152/33803 [00:01<07:50, 71.57it/s]/var/folders/rr/7lt3z7lx0dv_6xh6d53km6980000gn/T/ipykernel_33204/2691563409.py:35: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  events['CHARTTIME'] = pd.to_datetime(events['CHARTTIME'], errors='coerce')
  2%|▏         | 705/33803 [00:09<10:53, 50.63it/s]/var/folders/rr/7lt3z7lx0dv_6xh6d53km6980000gn/T/ipykernel_33204/2691563409.py:35: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  events['CHARTTIME'] = pd.to_datetime(events['CHARTTIME'], errors='coerce')
 18%|█▊        | 6127/33803 [01:29<05:48, 79.47it/s]/var/folders/rr/7lt3z7lx0dv_6xh6d53km6980000gn/T/ipykernel_33204/2691563409.py:36: UserWarning: Could not infer format, so each element will be parsed individ

In [22]:
events = pd.read_csv("/Users/william/Desktop/ERP/mimic3-benchmarks/data/root/3/events.csv")
events

,SUBJECT_ID,HADM_ID,ICUSTAY_ID,CHARTTIME,ITEMID,VALUE,VALUEUOM
0,3,145834,211552,2101-10-25 03:00:00,861,10.7,NaN
1,3,145834,211552,2101-10-25 03:00:00,1046,No,NaN
2,3,145834,211552,2101-10-25 03:00:00,1087,Pt Verbalized,NaN
3,3,145834,211552,2101-10-25 03:00:00,1125,MICU,NaN
4,3,145834,211552,2101-10-25 03:00:00,1127,10.7,NaN
...,...,...,...,...,...,...,...
7712,3,145834,211552,2101-10-25 20:00:00,40055,160,ml
7713,3,145834,211552,2101-10-25 22:00:00,40055,120,ml
7714,3,145834,211552,2101-10-21 12:00:00,40055,35,ml
7715,3,145834,211552,2101-10-26 06:00:00,40055,80,ml


In [23]:
stays = pd.read_csv("/Users/william/Desktop/ERP/mimic3-benchmarks/data/root/3/stays.csv")
stays

,SUBJECT_ID,HADM_ID,ICUSTAY_ID,LAST_CAREUNIT,DBSOURCE,INTIME,OUTTIME,LOS,ADMITTIME,DISCHTIME,DEATHTIME,ETHNICITY,DIAGNOSIS,GENDER,DOB,DOD,AGE,MORTALITY_INUNIT,MORTALITY,MORTALITY_INHOSPITAL
0,3,145834,211552,MICU,carevue,2101-10-20 19:10:11,2101-10-26 20:43:09,6.0646,2101-10-20 19:08:00,2101-10-31 13:58:00,NaN,WHITE,HYPOTENSION,M,2025-04-11,2102-06-14,76.577531,0,0,0


In [3]:
three_csv  = pd.read_csv("/Users/william/Desktop/ERP/Data/Stage_2/3.csv")
three_csv

,SUBJECT_ID,HADM_ID,ICUSTAY_ID,CHARTTIME,LOS,AGE,GENDER,MORTALITY_INUNIT,MORTALITY_INHOSPITAL,ALT,...,Vancomycin/Random,Venous_CO2(Calc),Venous_PvCO2,Venous_PvO2,Venous_pH,Ventilator_No.,"WBC_(4-11,000)",Weight_Change,avDO2,calprevflg
0,3,145834,211552,2101-10-20 18:00:00,6.0646,76.577531,M,0,0,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1.0
1,3,145834,211552,2101-10-20 19:00:00,6.0646,76.577531,M,0,0,25.0,...,NaN,NaN,NaN,NaN,NaN,NaN,15.2,NaN,NaN,1.0
2,3,145834,211552,2101-10-20 20:00:00,6.0646,76.577531,M,0,0,NaN,...,NaN,NaN,NaN,NaN,NaN,7.0,NaN,NaN,NaN,1.0
3,3,145834,211552,2101-10-20 21:00:00,6.0646,76.577531,M,0,0,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1.0
4,3,145834,211552,2101-10-20 22:00:00,6.0646,76.577531,M,0,0,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
138,3,145834,211552,2101-10-26 13:00:00,6.0646,76.577531,M,0,0,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1.0
139,3,145834,211552,2101-10-26 14:00:00,6.0646,76.577531,M,0,0,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1.0
140,3,145834,211552,2101-10-26 15:00:00,6.0646,76.577531,M,0,0,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1.0
141,3,145834,211552,2101-10-26 16:00:00,6.0646,76.577531,M,0,0,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [4]:
three_csv['Urine_Out_Foley']

0        NaN
1        NaN
2        NaN
3        NaN
4        NaN
       ...  
138    200.0
139     80.0
140    200.0
141     45.0
142     60.0
Name: Urine_Out_Foley, Length: 143, dtype: float64

In [2]:
import sys
!{sys.executable} -m pip install pyarrow


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 30.8/30.8 MB 27.0 MB/s eta 0:00:00a 0:00:01

[notice] A new release of pip is available: 25.0.1 -> 25.1.1
[notice] To update, run: pip3 install --upgrade pip


In [ ]:
import polars as pl
import os
from glob import glob
from tqdm import tqdm
from concurrent.futures import ThreadPoolExecutor, as_completed

# Input/output paths
stage2_dir = "/Users/william/Desktop/ERP/Data/Stage_2"
output_path = "/Users/william/Desktop/ERP/Data/merged_data.csv"
csv_files = glob(os.path.join(stage2_dir, "*.csv"))

# Column mapping: standardized_name -> actual_column_name_in_files
column_mapping = {
    'SUBJECT_ID': 'SUBJECT_ID',
    'HADM_ID': 'HADM_ID',
    'ICUSTAY_ID': 'ICUSTAY_ID',
    'CHARTTIME': 'CHARTTIME',
    'LOS': 'LOS',
    'GENDER': 'GENDER',
    'AGE': 'AGE',
    'Heart Rate': 'Heart_Rate',
    'O2 Saturation': 'SpO2',
    'Respiratory Rate': 'Respiratory_Rate',
    'Temperature (C)': 'Temperature_C',
    'Mean Blood Pressure': 'Arterial_BP_Mean',
    'Systolic Blood Pressure': 'Arterial_BP_[Systolic]',
    'Diastolic Blood Pressure': 'Arterial_BP_[Diastolic]',
    'White Blood Cells': 'White_Blood_Cells',
    'WBC': 'WBC_(4-11,000)',
    'pH': 'Arterial_pH',
    'Glucose': 'Glucose_(70-105)',
    'Sodium': 'Sodium_(135-148)',
    'Potassium': 'Potassium_(3.5-5.3)',
    'Creatinine': 'Creatinine_(0-1.3)',
    'BUN': 'BUN_(6-20)',
    'Hematocrit': 'Hematocrit',
    'Height': 'Height',
    'Weight': 'Daily_Weight',
    'GCS Total': 'GCS_Total',
    'Lactic Acid': 'Lactic_Acid(0.5-2.0)',
    'FiO2': 'FiO2_Set',
    'PaO2': 'Arterial_PaO2',
    'SpO2': 'SpO2',
    'MORTALITY_INUNIT': 'MORTALITY_INUNIT',
    'MORTALITY_INHOSPITAL': 'MORTALITY_INHOSPITAL'
}

# Define expected output schema
target_columns = {
    key: (pl.Utf8 if "ID" in key or key in ["CHARTTIME", "GENDER", "MORTALITY_INUNIT", "MORTALITY_INHOSPITAL"] else pl.Float64)
    for key in column_mapping
}

# Function to load and clean a single CSV
def load_and_clean_csv(file):
    try:
        df = pl.read_csv(file)

        # Add missing columns as nulls
        for std_col, actual_col in column_mapping.items():
            if actual_col not in df.columns:
                df = df.with_columns(pl.lit(None, dtype=target_columns[std_col]).alias(actual_col))

        # Select and rename
        df = df.select([
            pl.col(actual_col).cast(target_columns[std_col]).alias(std_col)
            for std_col, actual_col in column_mapping.items()
        ])

        return df
    except Exception as e:
        print(f"⚠️ Failed on {file}: {e}")
        return None

# Load CSVs in parallel
df_list = []
with ThreadPoolExecutor(max_workers=os.cpu_count()) as executor:
    futures = {executor.submit(load_and_clean_csv, f): f for f in csv_files}
    for future in tqdm(as_completed(futures), total=len(futures)):
        result = future.result()
        if result is not None:
            df_list.append(result)

# Merge and save
if df_list:
    df_all = pl.concat(df_list, how='diagonal_relaxed')
    df_all.write_csv(output_path)
    print(f"Merged file saved to: {output_path}")
else:
    print("No dataframes were loaded successfully.")


In [3]:
import pandas as pd

In [40]:
merged_data = pd.read_csv("/Users/william/Desktop/ERP/Data/merged_data.csv")
merged_data

,SUBJECT_ID,HADM_ID,ICUSTAY_ID,CHARTTIME,LOS,GENDER,AGE,Heart Rate,O2 Saturation,Respiratory Rate,...,Glucose,Sodium,Potassium,Creatinine,BUN,Hematocrit,Height,Weight,MORTALITY_INUNIT,MORTALITY_INHOSPITAL
0,30429,191715,290749,2160-01-07 10:00:00,3.1032,M,84.318990,NaN,NaN,NaN,...,143.0,NaN,5.8,NaN,NaN,22.000000,NaN,NaN,0,0
1,30429,191715,290749,2160-01-07 11:00:00,3.1032,M,84.318990,NaN,NaN,NaN,...,177.0,134.0,5.7,NaN,NaN,21.566667,NaN,NaN,0,0
2,30429,191715,290749,2160-01-07 12:00:00,3.1032,M,84.318990,NaN,NaN,NaN,...,120.0,135.0,4.5,NaN,NaN,22.450000,NaN,NaN,0,0
3,30429,191715,290749,2160-01-07 13:00:00,3.1032,M,84.318990,89.666667,100.00,NaN,...,109.0,135.0,4.9,0.8,17.0,25.200001,NaN,78.0,0,0
4,30429,191715,290749,2160-01-07 14:00:00,3.1032,M,84.318990,90.750000,97.25,13.666667,...,143.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
3497831,70772,171802,236849,2200-04-10 07:00:00,2.0595,M,86.398396,77.000000,NaN,11.000000,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0,0
3497832,70772,171802,236849,2200-04-10 08:00:00,2.0595,M,86.398396,81.000000,NaN,11.000000,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0,0
3497833,70772,171802,236849,2200-04-10 09:00:00,2.0595,M,86.398396,84.000000,NaN,15.000000,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0,0
3497834,70772,171802,236849,2200-04-10 10:00:00,2.0595,M,86.398396,83.000000,NaN,14.000000,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0,0


In [41]:
merged_data.isna().sum()

SUBJECT_ID                        0
HADM_ID                           0
ICUSTAY_ID                        0
CHARTTIME                         0
LOS                             998
GENDER                            0
AGE                               0
Heart Rate                   143859
O2 Saturation               1634878
Respiratory Rate             210691
Temperature (C)             3318039
Mean Blood Pressure         2428659
Systolic Blood Pressure     2422188
Diastolic Blood Pressure    2422219
White Blood Cells           3497836
WBC                         3377916
pH                          3318002
Glucose                     3259671
Sodium                      3349024
Potassium                   3289975
Creatinine                  3364980
BUN                         3365559
Hematocrit                  3318763
Height                      3488431
Weight                      3436825
MORTALITY_INUNIT                  0
MORTALITY_INHOSPITAL              0
dtype: int64